In [73]:
# 02_nb is for creating and cleaning one reference csv for each species' atm protein sequence

In [74]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('c:/Users/Dell - i5 11th Gen/Desktop/atm-protein-conservation-explorer')

In [75]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

fasta_file = (
    project_root
    / "data"
    / "raw"
    / "atm_vertebrate_orthologs"
    / "ncbi_dataset"
    / "data"
    / "protein.faa"
)

records = list(SeqIO.parse(fasta_file, "fasta"))
print("ID:", records[0].id)
print("Description:", records[0].description)
print("Sequence length:", len(records[0].seq))
print("First 50 amino acids:", records[0].seq[:50])




ID: XP_040832827.1
Description: XP_040832827.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
Sequence length: 3059
First 50 amino acids: MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRHSDSK


In [76]:
print("Total protein records:", len(records))

for record in records[:10]:
    print(record.description)

Total protein records: 2603
XP_040832827.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
XP_040832828.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X1]
XP_040832829.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X2]
XP_040832830.1 ATM [organism=Ochotona curzoniae] [GeneID=121154090] [isoform=X3]
XP_059322021.1 ATM [organism=Ammospiza nelsoni] [GeneID=132069672]
XP_079784892.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=1]
XP_079784893.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=2]
XP_079784894.1 ATM [organism=Stenella coeruleoalba] [GeneID=307190021] [isoform=3]
XP_084165020.1 atm [organism=Notolepis coatsorum] [GeneID=311224077]
XP_044525056.1 ATM [organism=Gracilinanus agilis] [GeneID=123241475] [isoform=X1]


In [77]:
import re

organisms = []

# finding length
for record in records:
    match = re.search(r"\[organism=([^\]]+)\]", record.description)

    if match:
        organisms.append(match.group(1))

print("Protein records:", len(records))
print("Unique species:", len(set(organisms)))


Protein records: 2603
Unique species: 849


In [78]:
import pandas as pd


def extract_header_field(description, field):
    pattern = rf"\[{field}=([^\]]+)\]"
    match = re.search(pattern, description)

    if match:
        return match.group(1)

    return pd.NA


rows = []

# fasta into list of dicts
for record in records:
    description_parts = record.description.split(maxsplit=2)
    sequence = str(record.seq).upper()

    row = {
        "accession": record.id,
        "gene_symbol": description_parts[1],
        "organism": extract_header_field(
            record.description,
            "organism"
        ),
        "gene_id": extract_header_field(
            record.description,
            "GeneID"
        ),
        "isoform": extract_header_field(
            record.description,
            "isoform"
        ),
        "sequence_length": len(sequence),
        "ambiguous_x_count": sequence.count("X"),
        "stop_count": sequence.count("*"),
        "sequence": sequence,
    }

    rows.append(row)    


#list of dicts to df
candidate_df = pd.DataFrame(rows)

candidate_df.head()

,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence
0,XP_040832827.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...
1,XP_040832828.1,ATM,Ochotona curzoniae,121154090,X1,3059,0,0,MSLALNDLLICCRHLEHERATERRKEVEKFKRLIRDPETIQQLDRH...
2,XP_040832829.1,ATM,Ochotona curzoniae,121154090,X2,2935,0,0,METVKDSSNSSTYGADCSNILLKDILSVRKYWCEISQQQWLELFSV...
3,XP_040832830.1,ATM,Ochotona curzoniae,121154090,X3,2623,0,0,MILYQLLSQRRRGERTPYVLRCLTEVALCQGKKSNLETSQQSDLLK...
4,XP_059322021.1,ATM,Ammospiza nelsoni,132069672,<NA>,3066,0,0,MSLALHDLLVCCRRLENEKAVERRKEIENFKRLLRDPETVLQLDRN...


In [79]:
def get_accession_priority(accession):
    if accession.startswith("NP_"):
        return 0

    if accession.startswith("XP_"):
        return 1

    return 2


human_reference = candidate_df[
    candidate_df["accession"] == "NP_000042.3"
]

if human_reference.empty:
    raise ValueError(
        "Human reference NP_000042.3 was not found."
    )


human_length = human_reference.iloc[0]["sequence_length"]

print("Human ATM reference length:", human_length)


candidate_df["accession_priority"] = (
    candidate_df["accession"]
    .apply(get_accession_priority)
)
candidate_df["length_difference"] = (
    candidate_df["sequence_length"] - human_length
).abs()

candidate_df["length_ratio"] = (
    candidate_df["sequence_length"] / human_length
)


# Start by marking every protein as a length failure
candidate_df["length_status"] = "Fail"


# Within 75%–125%: unusual but worth reviewing
candidate_df.loc[
    candidate_df["length_ratio"].between(0.75, 1.25),
    "length_status"] = "Review"


# Within 90%–110%: normal-looking ATM length
candidate_df.loc[
    candidate_df["length_ratio"].between(0.90, 1.10),
    "length_status"] = "Pass"


candidate_df["passes_basic_qc"] = (
    candidate_df["organism"].notna()
    & (candidate_df["gene_symbol"].str.upper() == "ATM")
    & (candidate_df["length_status"] != "Fail")
    & (candidate_df["ambiguous_x_count"] <= 5)
    & (candidate_df["stop_count"] == 0)
)


print(candidate_df["length_status"].value_counts())

Human ATM reference length: 3056
length_status
Pass      2006
Fail       430
Review     167
Name: count, dtype: int64


In [80]:
# separate df for selected rows
selected_df = (
    candidate_df.loc[candidate_df["passes_basic_qc"]]
    .sort_values(
        by=[
            "organism",
            "accession_priority",
            "ambiguous_x_count",
            "length_difference",
        ]
    )
    .drop_duplicates(
        subset=["organism"],
        keep="first",
    )
    .reset_index(drop=True)
    .copy()
)

print("Candidate proteins:", len(candidate_df))
print("Species before QC:", candidate_df["organism"].nunique())
print("Selected species:", selected_df["organism"].nunique())
selected_df.head()

Candidate proteins: 2603
Species before QC: 849
Selected species: 827


,accession,gene_symbol,organism,gene_id,isoform,sequence_length,ambiguous_x_count,stop_count,sequence,accession_priority,length_difference,length_ratio,length_status,passes_basic_qc
0,XP_083981534.1,atm,Abramis brama,311090226,<NA>,3102,0,0,MSLALHELLLCCRGLENEKATERKKEVDRFRRLIRSPDTVEELDRT...,1,46,1.015052,Pass,True
1,XP_022052894.2,atm,Acanthochromis polyacanthus,110953284,X1,3092,0,0,MSLALNDLLLCCRGLESDKATERKKETERFRLLLQSPETIQELDRT...,1,36,1.011780,Pass,True
2,XP_036940904.1,atm,Acanthopagrus latus,119011683,<NA>,3090,0,0,MSLALHDLMVCCRGLESDKATERKKVAEQFRRLIRTPETVQELDRS...,1,34,1.011126,Pass,True
3,XP_026892237.2,ATM,Acinonyx jubatus,106983963,X3,3057,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIRDPETVQHLDRH...,1,1,1.000327,Pass,True
4,XP_084640426.1,ATM,Acomys minous,311598838,1,3054,0,0,MSLALNDLLICCRQLEHDRATERRKEVEKFKRLIQDPETVQNLDRH...,1,2,0.999346,Pass,True


In [81]:

candidate_output = (
    project_root
    / "data"
    / "processed"
    / "atm_ortholog_candidates.csv"
)

selected_output = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.csv"
)

candidate_df.to_csv(candidate_output, index=False)
selected_df.to_csv(selected_output, index=False)

print(f"Candidates saved to: {candidate_output}")
print(f"Selected saved to: {selected_output}")

Candidates saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_ortholog_candidates.csv
Selected saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_representative_proteins.csv


In [82]:
# recheck if human_reference survived filtering
human_reference = selected_df[
    selected_df["accession"] == "NP_000042.3"
]

print("Human reference records found:", len(human_reference))

if human_reference.empty:
    raise ValueError(
        "Human ATM reference NP_000042.3 was not selected."
    )


Human reference records found: 1


In [83]:
selected_records = []

for row in selected_df.itertuples(index=False):
    description = (
        f"ATM "
        f"[organism={row.organism}] "
        f"[GeneID={row.gene_id}] "
        f"[isoform={row.isoform}]"
    )

    fasta_record = SeqRecord(
        seq=Seq(row.sequence),
        id=row.accession,
        description=description,
    )

    selected_records.append(fasta_record)

In [84]:
fasta_output = (
    project_root
    / "data"
    / "processed"
    / "atm_representative_proteins.fasta"
)

written_count = SeqIO.write(
    selected_records,
    fasta_output,
    "fasta",
)

print(f"Wrote {written_count} protein sequences.")
print(f"Saved to: {fasta_output}")

Wrote 827 protein sequences.
Saved to: c:\Users\Dell - i5 11th Gen\Desktop\atm-protein-conservation-explorer\data\processed\atm_representative_proteins.fasta


In [85]:
saved_records = list(
    SeqIO.parse(fasta_output, "fasta")
)

print("Expected records:", len(selected_df))
print("Saved FASTA records:", len(saved_records))
print("First saved header:", saved_records[0].description)

Expected records: 827
Saved FASTA records: 827
First saved header: XP_083981534.1 ATM [organism=Abramis brama] [GeneID=311090226] [isoform=<NA>]
